In [7]:
!nvidia-smi

Mon Sep 14 09:03:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
%cd /kaggle/working

!rm -rf sign-language-recognition

!git clone https://github.com/ashu123243/sign-language-recognition.git

%cd /kaggle/working/sign-language-recognition

/kaggle/working
Cloning into 'sign-language-recognition'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 222 (delta 112), reused 189 (delta 79), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 213.14 KiB | 2.09 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/kaggle/working/sign-language-recognition


In [9]:
!pip install -q -e .

  Preparing metadata (setup.py) ... done


In [10]:
import sys
import os

sys.path.insert(
    0,
    "/kaggle/working/sign-language-recognition/src"
)

os.environ["PYTHONPATH"] = (
    "/kaggle/working/sign-language-recognition/src"
)

print("Project:", os.getcwd())
print("Source path configured")

Project: /kaggle/working/sign-language-recognition
Source path configured


In [11]:
!git status
!git log -1 --oneline

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	src/sign_language_detection.egg-info/

nothing added to commit but untracked files present (use "git add" to track)
aae893f (HEAD -> main, origin/main, origin/HEAD) update model architecture


In [12]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 2:
        print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ngphmng


In [13]:
!find /kaggle/input -type f | head -50

/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val.csv
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer1_sample1164_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer18_sample598_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer35_sample287_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer1_sample834_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer35_sample338_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer18_sample273_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer1_sample1100_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer25_sample83_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer25_sample536_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer11_sample422_color.mp4
/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val/signer1_sample315_color.mp4
/kaggle/input/datase

In [14]:
from pathlib import Path

project_root = Path("/kaggle/working/sign-language-recognition")
raw_dir = project_root / "data" / "raw"

raw_dir.mkdir(parents=True, exist_ok=True)

dataset_root = Path(
    "/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL"
)

# AUTSL video directory
autsl_link = raw_dir / "AUTSL"

if not autsl_link.exists():
    autsl_link.symlink_to(dataset_root, target_is_directory=True)

# CSV files
for filename in ["train.csv", "val.csv", "test.csv"]:

    source = dataset_root / filename
    target = raw_dir / filename

    if not target.exists():
        target.symlink_to(source)

print("Dataset linked successfully.")
print("Dataset root :", dataset_root)
print("Project raw  :", raw_dir)

Dataset linked successfully.
Dataset root : /kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL
Project raw  : /kaggle/working/sign-language-recognition/data/raw


In [15]:
from pathlib import Path

project_root = Path("/kaggle/working/sign-language-recognition")

print("Train CSV:", (project_root / "data/raw/train.csv").exists())
print("Val CSV  :", (project_root / "data/raw/val.csv").exists())
print("Test CSV :", (project_root / "data/raw/test.csv").exists())

print(
    "Train dir:",
    (project_root / "data/raw/AUTSL/train").exists()
)

print(
    "Val dir  :",
    (project_root / "data/raw/AUTSL/val").exists()
)

print(
    "Test dir :",
    (project_root / "data/raw/AUTSL/test").exists()
)

Train CSV: True
Val CSV  : True
Test CSV : True
Train dir: False
Val dir  : False
Test dir : False


In [16]:
from pathlib import Path

dataset_root = Path(
    "/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL"
)

print("Dataset root contents:")
for item in sorted(dataset_root.iterdir()):
    print(
        f"{'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}"
    )

Dataset root contents:
[DIR]  test
[FILE] test.csv
[DIR]  train
[FILE] train.csv
[DIR]  val
[FILE] val.csv


In [17]:
print("\nVideo directories:")

for name in ["train", "val", "test"]:
    path = dataset_root / name
    print(
        name,
        "->",
        path,
        "| exists:",
        path.exists(),
        "| is_dir:",
        path.is_dir()
    )


Video directories:
train -> /kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/train | exists: True | is_dir: True
val -> /kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/val | exists: True | is_dir: True
test -> /kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL/test | exists: True | is_dir: True


In [18]:
from pathlib import Path
import shutil

project_root = Path(
    "/kaggle/working/sign-language-recognition"
)

raw_dir = project_root / "data" / "raw"

dataset_root = Path(
    "/kaggle/input/datasets/ngphmng/autsl-dataset/AUTSL"
)

autsl_link = raw_dir / "AUTSL"


# Remove existing AUTSL directory/symlink
if autsl_link.is_symlink():
    autsl_link.unlink()

elif autsl_link.exists():
    shutil.rmtree(autsl_link)


# Create correct symlink
autsl_link.symlink_to(
    dataset_root,
    target_is_directory=True
)


# Recreate CSV symlinks
for filename in [
    "train.csv",
    "val.csv",
    "test.csv"
]:

    target = raw_dir / filename

    if target.is_symlink():
        target.unlink()

    elif target.exists():
        target.unlink()

    target.symlink_to(
        dataset_root / filename
    )


print("Correct dataset links created.")
print("AUTSL:", autsl_link)

Correct dataset links created.
AUTSL: /kaggle/working/sign-language-recognition/data/raw/AUTSL


In [19]:
from pathlib import Path

root = Path(
    "/kaggle/working/sign-language-recognition"
)

checks = {
    "Train CSV":
        root / "data/raw/train.csv",

    "Val CSV":
        root / "data/raw/val.csv",

    "Test CSV":
        root / "data/raw/test.csv",

    "Train videos":
        root / "data/raw/AUTSL/train",

    "Val videos":
        root / "data/raw/AUTSL/val",

    "Test videos":
        root / "data/raw/AUTSL/test",
}

for name, path in checks.items():

    print(
        f"{name:15} : "
        f"{path.exists()} | "
        f"{path}"
    )

Train CSV       : True | /kaggle/working/sign-language-recognition/data/raw/train.csv
Val CSV         : True | /kaggle/working/sign-language-recognition/data/raw/val.csv
Test CSV        : True | /kaggle/working/sign-language-recognition/data/raw/test.csv
Train videos    : True | /kaggle/working/sign-language-recognition/data/raw/AUTSL/train
Val videos      : True | /kaggle/working/sign-language-recognition/data/raw/AUTSL/val
Test videos     : True | /kaggle/working/sign-language-recognition/data/raw/AUTSL/test


In [20]:
from pathlib import Path

train_dir = Path(
    "/kaggle/working/sign-language-recognition/data/raw/AUTSL/train"
)

videos = list(
    train_dir.glob("*.mp4")
)

print("Train videos found:", len(videos))

if videos:
    print("First video:", videos[0])

Train videos found: 28142
First video: /kaggle/working/sign-language-recognition/data/raw/AUTSL/train/signer41_sample595_color.mp4


In [21]:
%cd /kaggle/working/sign-language-recognition

!PYTHONPATH=src python -c "from sign_language_detection.configuration import ConfigurationManager; from sign_language_detection.components.data_ingestion import DataIngestion; c=ConfigurationManager().get_data_ingestion_config(); d=DataIngestion(c); print(d.initiate_data_ingestion())"

/kaggle/working/sign-language-recognition
(PosixPath('/kaggle/working/sign-language-recognition/artifacts/data/train_processed.csv'), PosixPath('/kaggle/working/sign-language-recognition/artifacts/data/val_processed.csv'), PosixPath('/kaggle/working/sign-language-recognition/artifacts/data/test_processed.csv'))


In [22]:
from pathlib import Path
import pandas as pd

root = Path("/kaggle/working/sign-language-recognition")

for name in ["train", "val", "test"]:
    path = root / f"artifacts/data/{name}_processed.csv"
    df = pd.read_csv(path)

    print(
        f"{name:5} | "
        f"samples: {len(df):5} | "
        f"classes: {df['label'].nunique()}"
    )

train | samples: 28142 | classes: 226
val   | samples:  4418 | classes: 226
test  | samples:  3742 | classes: 226


In [23]:
%cd /kaggle/working/sign-language-recognition

!PYTHONPATH=src python -c "from sign_language_detection.configuration import ConfigurationManager; from sign_language_detection.components.data_validation import DataValidation; c=ConfigurationManager().get_data_validation_config(); v=DataValidation(c); print('Validation result:', v.initiate_data_validation())"

/kaggle/working/sign-language-recognition
Validation result: True


In [24]:
%cd /kaggle/working/sign-language-recognition

import sys
import torch

sys.path.insert(
    0,
    "/kaggle/working/sign-language-recognition/src"
)

from sign_language_detection.configuration import ConfigurationManager
from sign_language_detection.components.model_trainer import ModelTrainer


config = ConfigurationManager().get_model_trainer_config()

trainer = ModelTrainer(config)

train_loader, _, _, _ = trainer._create_dataloaders()

model = trainer._build_model()

criterion, optimizer, scheduler = trainer._compile_model(model)


videos, labels = next(iter(train_loader))

print("Input shape :", videos.shape)
print("Labels shape:", labels.shape)
print("Device      :", trainer.device)


videos = videos.to(
    trainer.device,
    non_blocking=True
)

labels = labels.to(
    trainer.device,
    non_blocking=True
)


model.train()

optimizer.zero_grad(
    set_to_none=True
)


with torch.amp.autocast(
    device_type="cuda",
    enabled=trainer.use_amp
):

    logits = model(videos)

    loss = criterion(
        logits,
        labels
    )


trainer.scaler.scale(
    loss
).backward()


trainer.scaler.unscale_(
    optimizer
)


torch.nn.utils.clip_grad_norm_(
    [
        p
        for p in model.parameters()
        if p.requires_grad
    ],
    config.gradient_clip_value
)


trainer.scaler.step(
    optimizer
)

trainer.scaler.update()


print("\n--- Sanity Test Passed ---")
print("Output shape:", logits.shape)
print("Loss        :", loss.item())
print("Backward    : OK")
print("Optimizer   : OK")
print("AMP         : OK")

/kaggle/working/sign-language-recognition
Input shape : torch.Size([4, 3, 8, 160, 160])
Labels shape: torch.Size([4])
Device      : cuda

--- Sanity Test Passed ---
Output shape: torch.Size([4, 226])
Loss        : 5.82113790512085
Backward    : OK
Optimizer   : OK
AMP         : OK


In [25]:
%cd /kaggle/working/sign-language-recognition

!PYTHONPATH=src python main.py

/kaggle/working/sign-language-recognition

Data Ingestion Completed Successfully
Train metadata      : /kaggle/working/sign-language-recognition/artifacts/data/train_processed.csv
Validation metadata : /kaggle/working/sign-language-recognition/artifacts/data/val_processed.csv
Test metadata       : /kaggle/working/sign-language-recognition/artifacts/data/test_processed.csv

Data Validation Completed Successfully
Validation result : True

Data Transformation Completed Successfully
Transformation result : True

Epoch 1/12
Epoch 1/12 [Train] | Batch 1383/7036[mov,mp4,m4a,3gp,3g2,mj2 @ 0x27a8ab00] moov atom not found
Epoch 1/12 [Train] | Batch 1559/7036[mov,mp4,m4a,3gp,3g2,mj2 @ 0x27a9d1c0] moov atom not found
Epoch 1/12 [Train] | Batch 3301/7036[mov,mp4,m4a,3gp,3g2,mj2 @ 0x27e2e700] moov atom not found
Epoch 1/12 [Train] | Batch 3435/7036[mov,mp4,m4a,3gp,3g2,mj2 @ 0x27a79580] moov atom not found
Epoch 1/12 [Train] | Batch 3502/7036[mov,mp4,m4a,3gp,3g2,mj2 @ 0x27a94c80] moov atom not found


In [26]:
!find /kaggle/working/sign-language-recognition/artifacts -type f -printf '%p  %k KB\n' | sort

/kaggle/working/sign-language-recognition/artifacts/checkpoints/best_model.pth  374288 KB
/kaggle/working/sign-language-recognition/artifacts/data/test_processed.csv  120 KB
/kaggle/working/sign-language-recognition/artifacts/data/train_processed.csv  888 KB
/kaggle/working/sign-language-recognition/artifacts/data/val_processed.csv  140 KB
/kaggle/working/sign-language-recognition/artifacts/model/final_results.csv  4 KB
/kaggle/working/sign-language-recognition/artifacts/model/sign_language_model.pth  130092 KB


In [27]:
!cd /kaggle/working/sign-language-recognition && zip -r sign_language_model_backup.zip artifacts/

  adding: artifacts/ (stored 0%)
  adding: artifacts/data/ (stored 0%)
  adding: artifacts/data/test_processed.csv (deflated 86%)
  adding: artifacts/data/train_processed.csv (deflated 86%)
  adding: artifacts/data/val_processed.csv (deflated 86%)
  adding: artifacts/model/ (stored 0%)
  adding: artifacts/model/final_results.csv (deflated 36%)
  adding: artifacts/model/sign_language_model.pth (deflated 7%)
  adding: artifacts/checkpoints/ (stored 0%)
  adding: artifacts/checkpoints/best_model.pth (deflated 36%)


In [28]:
!ls -lh /kaggle/working/sign-language-recognition/sign_language_model_backup.zip

-rw-r--r-- 1 root root 352M Sep 14 15:11 /kaggle/working/sign-language-recognition/sign_language_model_backup.zip


In [29]:

!cp /kaggle/working/sign-language-recognition/sign_language_model_backup.zip /kaggle/working/

In [30]:
!curl https://rclone.org/install.sh | sudo bash

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4734  100  4734    0     0   8769      0 --:--:-- --:--:-- --:--:--  8782
Archive:  rclone-current-linux-amd64.zip
   creating: tmp_unzip_dir_for_rclone/rclone-v1.75.1-linux-amd64/
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.1-linux-amd64/README.txt  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.1-linux-amd64/rclone.1  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.1-linux-amd64/rclone  [binary]
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.1-linux-amd64/git-log.txt  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.1-linux-amd64/README.html  [text]  
Purging old database entries in /usr/share/man...
Processing manual pages under /usr/share/man...
Purging old database entries in /usr/share/man/pl...
Processing manual pages under /usr/share/man/pl...
Purging old database entries

In [31]:
!rclone version

rclone v1.75.1
- os/version: ubuntu 22.04 (64 bit)
- os/kernel: 6.12.90+ (x86_64)
- os/type: linux
- os/arch: amd64
- go/version: go1.26.8
- go/linking: static
- go/tags: none


In [ ]:
!rclone config

2026/09/14 15:19:16 NOTICE: Config file "/root/.config/rclone/rclone.conf" not found - using defaults
No remotes found, make a new one?
n) New remote
s) Set configuration password
q) Quit config
n/s/q> 

In [40]:
from pathlib import Path

project_root = Path("/kaggle/working/sign-language-recognition")

for p in [
    project_root / "artifacts/checkpoints/best_model.pth",
    project_root / "artifacts/model/sign_language_model.pth",
]:
    if p.exists():
        print(f"FOUND: {p}")
        print(f"SIZE : {p.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"NOT FOUND: {p}")

FOUND: /kaggle/working/sign-language-recognition/artifacts/checkpoints/best_model.pth
SIZE : 365.51 MB
FOUND: /kaggle/working/sign-language-recognition/artifacts/model/sign_language_model.pth
SIZE : 127.04 MB


In [41]:
!grep -nEi "resume|checkpoint|load_state|best_model|torch.load" \
/kaggle/working/sign-language-recognition/src/sign_language_detection/components/model_trainer.py

1259:    # Save Best Checkpoint
1262:    def _save_checkpoint(
1274:            checkpoint_dir = (
1275:                self.config.checkpoint_dir
1278:            checkpoint_dir.mkdir(
1284:            checkpoint = {
1310:                checkpoint,
1311:                self.config.checkpoint_file
1316:                f"Best checkpoint saved: "
1317:                f"{self.config.checkpoint_file}"
1323:                "Failed to save checkpoint"
1535:                    self._save_checkpoint(
1635:            # Load Best Checkpoint
1639:                self.config.checkpoint_file.exists()
1643:                    "Best checkpoint was not created."
1647:            checkpoint = torch.load(
1648:                self.config.checkpoint_file,
1654:            model.load_state_dict(
1655:                checkpoint[


In [42]:
!sed -n '1250,1330p' /kaggle/working/sign-language-recognition/src/sign_language_detection/components/model_trainer.py

print("\n" + "="*80 + "\n")

!sed -n '1600,1670p' /kaggle/working/sign-language-recognition/src/sign_language_detection/components/model_trainer.py

        return (
            average_loss,
            accuracy,
            macro_f1,
            top5_accuracy
        )


    # ========================================================
    # Save Best Checkpoint
    # ========================================================

    def _save_checkpoint(
        self,
        model,
        optimizer,
        scheduler,
        epoch,
        best_val_f1,
        history
    ):

        try:

            checkpoint_dir = (
                self.config.checkpoint_dir
            )

            checkpoint_dir.mkdir(
                parents=True,
                exist_ok=True
            )


            checkpoint = {

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "best_val_f

In [43]:
!sed -n '1330,1570p' \
/kaggle/working/sign-language-recognition/src/sign_language_detection/components/model_trainer.py



    # ========================================================
    # Training
    # ========================================================

    def _train_model(self):

        try:

            (
                train_loader,
                train_eval_loader,
                val_loader,
                test_loader
            ) = self._create_dataloaders()


            model = self._build_model()


            (
                criterion,
                optimizer,
                scheduler
            ) = self._compile_model(
                model
            )


            best_val_f1 = -1.0

            best_epoch = -1

            epochs_without_improvement = 0

            history = []


            # =================================================
            # Epoch Loop
            # =================================================

            for epoch in range(
                self.config.epochs
            ):

                start_time = time.time()


          

In [44]:
!sed -n '1470,1645p' \
/kaggle/working/sign-language-recognition/src/sign_language_detection/components/model_trainer.py


                    "train_loss":
                        train_loss,

                    "train_accuracy":
                        train_acc,

                    "train_macro_f1":
                        train_f1,

                    "train_top5_accuracy":
                        train_top5,

                    "val_loss":
                        val_loss,

                    "val_accuracy":
                        val_acc,

                    "val_macro_f1":
                        val_f1,

                    "val_top5_accuracy":
                        val_top5,

                    "lr_layer3":
                        current_lrs[0],

                    "lr_layer4":
                        current_lrs[1],

                    "lr_fc":
                        current_lrs[2],

                    "epoch_time_seconds":
                        epoch_time,
                }


                history.append(
                    epoch_result
                )


                im

In [45]:
!grep -n -A120 "def _compile_model" \
/kaggle/working/sign-language-recognition/src/sign_language_detection/components/model_trainer.py

877:    def _compile_model(
878-        self,
879-        model
880-    ):
881-
882-        try:
883-
884-            criterion = nn.CrossEntropyLoss(
885-                label_smoothing=
886-                self.config.label_smoothing
887-            )
888-
889-
890-            optimizer = optim.AdamW(
891-                [
892-                    {
893-                        "params":
894-                            model.layer3.parameters(),
895-
896-                        "lr":
897-                            self.config.layer3_learning_rate,
898-                    },
899-
900-                    {
901-                        "params":
902-                            model.layer4.parameters(),
903-
904-                        "lr":
905-                            self.config.layer4_learning_rate,
906-                    },
907-
908-                    {
909-                        "params":
910-                            model.fc.parameters(),
911-
912-                        "